In [2]:
import json

import pandas as pd
from ml_enhance import get_preprocessed_smiles

In [3]:
DATA_SOURCE = "../data/AqSolDB/data_curated.csv"

In [4]:
def molecule_repr(id: str, smiles: str) -> dict:
    return {"id": id, "smiles": smiles}


def generate_json(inputs: list[dict]) -> dict:
    input_file = {
        "options": {
            "max_conformers": 32,
            "max_microstates": 8,
            "metadynamics": False
        },
        "runtime_options": {
        "num_vcpus": 1,
        "batch_size": 5
    },
    }

    input_file.update({"inputs": inputs})

    return input_file

In [5]:
df = pd.read_csv(DATA_SOURCE)

# preprocess the smiles for the quantumFP program
preprocessed_smiles = [get_preprocessed_smiles(smiles) for smiles in df["SMILES"]]

# Filter out all None instances returned from preprocess_smiles
cleaned_smiles = list(filter(lambda x: x not in (None, 'salt', 'atom'), preprocessed_smiles))

[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:14] WARNING: not removing hydrogen atom without neighbors
[16:56:15] WARNING: not removing hydrogen atom without neighbors
[16:56:15] WARNING: not removing hydrogen atom without neighbors
[16:56:15] WARNING: not removing hydrogen atom without neighbors
[16:56:15] WARNING: not r

In [11]:
num_atoms = preprocessed_smiles.count('atom')
num_salts = preprocessed_smiles.count('salt')
num_invalid = preprocessed_smiles.count(None)
num_succes = len(cleaned_smiles)

print(f"Total molecules in AqSolDB: {len(preprocessed_smiles)}\nNumber of salts: {num_salts}\nNumber of atoms: {num_atoms}\nNumber of invalid SMILES: {num_invalid}\nNumber of valid (used) SMILES: {num_succes}")

Total molecules in AqSolDB: 9982
Number of salts: 1098
Number of atoms: 37
Number of invalid SMILES: 2
Number of valid (used) SMILES: 8845


In [23]:
[og for (og, smiles) in zip(df["SMILES"], preprocessed_smiles) if smiles is None]

['CC1=CC=C[NH++]([O-])[CH-]1', 'OC(=O)C1=C[NH++]([O-])[CH-]C=C1']

In [ ]:
inputs = [molecule_repr(idx, smiles) for idx, smiles in enumerate(cleaned_smiles)]

json_obj = generate_json(inputs)

with open("../data/QFP_input.json", "w") as f:
    json.dump(json_obj, f, indent=4)

Whilst 8845 molecules passed, only 8837 molecules were returned from QuatnumFP, the remaining 8 were because QFP couldn't handle an extra proton that should be there in solvated form, but keeps dissociating in vacuum optimization